# Phase 11.5: Targeted Winter/Smog Feature Investigation

**Scientific Question**: Can physically motivated winter/smog feature proxies (inversion risk, fog indicators, stagnation enhancement, seasonal emission proxy) reduce the remaining winter forecasting error without sacrificing the temporal robustness established by EXP-019?

**Methodology & Guardrails**:
- **Baseline Frozen**: EXP-019 (`PersistenceAwareHybridModel`) architecture retrained from scratch per fold.
- **Ablation Strategy**: Independent evaluation of each feature family before combining (`ABL-001` to `ABL-004`), plus combined (`ABL-005`), against untouched baseline (`ABL-000`).
- **Adoption Gate**: Strict 6-criterion evaluation requiring winter mean improvement without unacceptable regression in individual folds or non-winter regimes.
- **Test Set Discipline**: The 2025–2026 test partition (`X_test_v2 / y_test_v2`) was completely untouched.

| Ablation ID | Feature Family | Description |
|:---|:---|:---|
| **ABL-000** | Baseline | Existing 113 features (weather, pollutants, lags, ratios) |
| **ABL-001** | Family 1 (Thermal/Inversion Proxy) | `diurnal_temp_range_24h`, `inversion_risk_proxy`, `temp_drop_6h` |
| **ABL-002** | Family 2 (Fog/Mist Indicator) | `fog_proxy`, `fog_hours_rolling_24h` |
| **ABL-003** | Family 3 (Stagnation Enhancement) | `wind_stagnation_hours_12h/24h`, `pressure_stability_24h` |
| **ABL-004** | Family 4 (Seasonal-Emission Proxy) | `crop_burning_season`, `winter_emission_intensity` |
| **ABL-005** | Combined | All 4 candidate families combined |

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

with open('../data/models/walk_forward/ablation/ablation_report.json', encoding='utf-8') as f:
    report = json.load(f)

df = pd.read_csv('../data/models/walk_forward/ablation/ablation_summary.csv')
print(f"Loaded {len(df)} ablation-fold evaluations across {df['ablation_id'].nunique()} configurations.")

## 1. Overall RMSE Leaderboard Across Walk-Forward Folds

In [ ]:
pivot_rmse = df.pivot_table(index='ablation_id', columns='fold_id', values='overall_rmse', aggfunc='first')
fold_cols = ['fold_1_winter2021', 'fold_2_transition_summer2022', 'fold_3_monsoon2022', 'fold_4_winter2023']
pivot_rmse = pivot_rmse[fold_cols]
pivot_rmse.columns = ['F1 (Winter 2021)', 'F2 (Trans/Sum 2022)', 'F3 (Monsoon 2022)', 'F4 (Winter 2023)']
pivot_rmse['Mean All Folds'] = pivot_rmse.mean(axis=1)
pivot_rmse['Mean Winter (F1+F4)'] = (pivot_rmse['F1 (Winter 2021)'] + pivot_rmse['F4 (Winter 2023)']) / 2.0

desc_map = {}
for abl_id, d in report['ablations'].items():
    desc_map[abl_id] = d['config']['description']

pivot_display = pivot_rmse.copy()
pivot_display.insert(0, 'Description', [desc_map.get(idx, '') for idx in pivot_display.index])
print(pivot_display.round(2).to_string())

## 2. Incremental Contribution Analysis (Δ RMSE vs Baseline ABL-000)

Positive values indicate an improvement (reduction in RMSE), while negative values represent regression.

In [ ]:
base_rmse = pivot_rmse.loc['ABL-000']
delta_df = pd.DataFrame(index=pivot_rmse.index)

for col in pivot_rmse.columns:
    delta_df[f'Δ {col}'] = base_rmse[col] - pivot_rmse[col]  # Positive = better

delta_df.insert(0, 'Description', [desc_map.get(idx, '') for idx in delta_df.index])
print(delta_df.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: Mean Overall RMSE
ax = axes[0]
abl_ids = list(pivot_rmse.index)
means = pivot_rmse['Mean All Folds']
bars = ax.bar(abl_ids, means, color=['#1976D2', '#78909C', '#4CAF50', '#E57373', '#81C784', '#B0BEC5'], alpha=0.9)
ax.set_ylabel('Mean RMSE Across All 4 Folds')
ax.set_title('Overall Model Performance across Ablations\n(lower = better)')
ax.set_ylim(80, 85)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f"{bar.get_height():.2f}", ha='center', va='bottom', fontsize=9)

# Right: Winter Delta (F1 + F4 mean)
ax = axes[1]
winter_deltas = delta_df['Δ Mean Winter (F1+F4)'].drop('ABL-000')
colors = ['#4CAF50' if x > 0 else '#E53935' for x in winter_deltas]
bars2 = ax.bar(winter_deltas.index, winter_deltas, color=colors, alpha=0.85)
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_ylabel('Δ Winter RMSE (Positive = Improvement)')
ax.set_title('Winter/Smog Incremental Value (Δ vs ABL-000)\n(mean across F1 & F4)')
for bar in bars2:
    h = bar.get_height()
    va = 'bottom' if h >= 0 else 'top'
    ax.text(bar.get_x() + bar.get_width() / 2, h + (0.01 if h >= 0 else -0.02),
            f"{h:+.3f}", ha='center', va=va, fontsize=9)

plt.tight_layout()
plt.savefig('../data/models/walk_forward/ablation/winter_ablation_delta.png', dpi=150)
plt.show()

## 3. Adoption Gate Summary (Criteria 1–6 Evaluation)

To guard against adopting features that improve one specific regime at the expense of others, every candidate must meet all 6 criteria.

In [ ]:
gate_rows = []
for abl_id, abl_data in report['ablations'].items():
    if abl_id == 'ABL-000':
        continue
    dec = abl_data.get('adoption_decision', {})
    crits = dec.get('criteria', {})
    gate_rows.append({
        'Ablation': abl_id,
        'Description': abl_data['config']['description'],
        '1. Winter Mean Δ': f"{crits.get('1_winter_mean_improvement', {}).get('mean_delta', 0):+.3f} (Passed: {crits.get('1_winter_mean_improvement', {}).get('passed')})",
        '2. Overall Δ': f"{crits.get('2_overall_no_regression', {}).get('delta', 0):+.3f} (Passed: {crits.get('2_overall_no_regression', {}).get('passed')})",
        '3. Summer/Monsoon': f"Passed: {crits.get('3_summer_monsoon_stable', {}).get('passed')}",
        '4. Horizons': f"Passed: {crits.get('4_horizon_stable', {}).get('passed')}",
        '5. Extremes': f"Passed: {crits.get('5_extreme_events_stable', {}).get('passed')}",
        '6. Naive Margin': f"{crits.get('6_naive_advantage_maintained', {}).get('mean_margin', 0):.1f} pts (Passed: {crits.get('6_naive_advantage_maintained', {}).get('passed')})",
        'Recommendation': dec.get('recommendation'),
    })

gate_df = pd.DataFrame(gate_rows)
print(gate_df.to_string(index=False))

## 4. Key Scientific Findings & Physical Interpretations

### Finding 1: Limited Incremental Value from Surface Proxies
- The baseline `ABL-000` (113 features) already includes extensive meteorological variables, lags, rolling dispersion stats, and domain ratios (`combustion_index`, `stagnation_index`, `pm_ratio`, `nitrogen_ozone_ratio`, barometric tendencies).
- Across all six configurations, the mean RMSE ranges narrowly from **83.33 to 83.66** (a full spread of only **0.33 RMSE points**). The maximum improvement over baseline is **0.11 RMSE points** (83.44 → 83.33 with ABL-004).
- This indicates that surface-derived meteorological proxies are near their predictive limit for 72-hour AQI.

### Finding 2: Stagnation Enhancement Redundancy (Family 3)
- `ABL-003` (+ Stagnation Enhancement: rolling wind calm counts and pressure stability) caused slight regressions in mean RMSE (83.44 → 83.66, with F2 degrading by +0.35 and F3 degrading by +0.36).
- This supports the hypothesis that the new stagnation features are largely redundant with existing features like `stagnation_index` (`PM2.5 / (wind_speed + 0.5)`) and standard wind speed lags.

### Finding 3: Plausible Information Gaps Beyond Surface Telemetry
- `ABL-001` (Thermal/Inversion Proxy via diurnal temperature range and cooling rate) produced no meaningful gain in winter (-0.064 RMSE delta).
- **Physical Insight**: Surface-level temperature changes are a coarse indicator of nocturnal boundary-layer dynamics. True atmospheric inversions depend heavily on vertical temperature profiles (lapse rates between surface and 850/925 hPa) and planetary boundary layer (PBL) height. Surface proxies cannot measure vertical atmospheric structure or real-time regional emission dynamics (active fire counts). These represent plausible sources of additional predictive information that current surface telemetry cannot represent.

### Finding 4: Seasonal Emission Proxy (Family 4)
- `ABL-004` (+ Punjab crop burning calendar window) showed a modest +0.37 RMSE improvement on F1 (Winter 2021) and +0.139 mean winter improvement.
- However, because this is a static calendar window rather than real-time fire detections (e.g. MODIS/VIIRS FIRMS fire radiative power), the signal is coarse and yields only a marginal ~0.11 point overall reduction.

## 5. Production Decision (Outcome B)

> **DECISION: EXP-019 REMAINS THE VALIDATED PRODUCTION CHAMPION.**

> **EXP-019 with the existing 113-feature schema remains the production champion. None of the tested winter/smog feature families satisfied the predefined adoption criteria. The best observed mean RMSE reduction was approximately 0.11 points, while some configurations introduced regressions in individual folds or non-winter regimes. Therefore, the additional feature complexity is not justified by the observed walk-forward performance.**